We are creating a method of compiling, accessing and analyzing data that represents geophysical, cosmic, and solar events. 
We want to use these events sensing the ionosphere to make predictions, explore correlations between variables, and more deeply understand space weather phenomenon.


we want to help the world with data.

In [ ]:
import plotly.graph_objs as go
import geopandas as gpd
import xarray as xr
import numpy as np
import plotly.graph_objs as go
import pandas as pd
from matplotlib import pyplot as plt
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.io as pio

In [ ]:
EFD = "CSES_data/CSES_01_EFD_1_L02_A1_213330_20211206_164953_20211206_172707_000.h5"
HEPPL = '/home/wvuser/DATA SET tutorial/webvalley2024_EP_tutorial/webvalley2024/WebValley 2024 Challenge 2/CSES_01_HEP_1_L02_A4_176401_20210407_182209_20210407_190029_000 (1).h5'

### Reduce Frequency of Data for _

In [ ]:
def reduce_frequency(data_array, frequency):
    """
    Reduce the number of measurements in each row of the xarray based on the specified frequency,
    while preserving the metadata.
    Parameters:
    data_array (xarray.DataArray): The input xarray.
    frequency (int): The number of elements to keep in each row.
    Returns:
    xarray.DataArray: The reduced xarray.
    """
    num_rows, num_cols = data_array.shape
    if frequency > num_cols:
        raise ValueError("Frequency cannot be greater than the number of columns in the data array.")
    reduced_rows = []
    for row in data_array.values:
        indices = np.linspace(0, num_cols - 1, frequency, dtype=int)
        reduced_row = row[indices]
        reduced_rows.append(reduced_row)
    reduced_values = np.stack(reduced_rows, axis=0)
    reduced_data_array = xr.DataArray(reduced_values,
                                      dims=[data_array.dims[0], 'reduced_freq'],
                                      coords={data_array.dims[0]: data_array.coords[data_array.dims[0]],
                                              'reduced_freq': indices},
                                      attrs=data_array.attrs)
    return reduced_data_array
 

### Mapped Electron Field Detector 

In [ ]:
def plot_EFD_on_map(data, latitude, longitude):
    
    try:
        freq = data.shape[1]
    except IndexError:
        freq = 1

    measure = data.values.flatten()

    lon = longitude.values.flatten()
    lat = latitude.values.flatten()

    length_coord = len(lon)
    length_measure = len(measure)

    lon_extend = np.concatenate([np.linspace(lon[i], lon[i+1], freq, endpoint=False) for i in range(length_coord-1)])
    lon_extend = np.concatenate([lon_extend, np.linspace(lon[-2], lon[-1], freq)])

    lat_extend = np.concatenate([np.linspace(lat[i], lat[i+1], freq, endpoint=False) for i in range(length_coord-1)])
    lat_extend = np.concatenate([lat_extend, np.linspace(lat[-2], lat[-1], freq)])

    # Create a scatter plot on a map using Plotly
    fig = go.Figure()

    # Load world map using geopandas
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

    # Plot world map as background
    fig.add_trace(go.Choropleth(
        geojson=world.__geo_interface__,
        locations=world.index,
        z=np.zeros(len(world)),  # Dummy values for Choropleth trace
        colorscale=[[0, 'lightgrey'], [1, 'lightgrey']],
        hoverinfo='none',
        showscale=False
    ))

    # Scatter plot for data points
    fig.add_trace(go.Scattergeo(
        lon=lon_extend,
        lat=lat_extend,
        mode='markers',
        marker=dict(
            size=10,
            color=measure,
            colorscale='Viridis',
            colorbar=dict(title="V/m"),
            opacity=0.8,
            colorbar_thickness=20,
            colorbar_x=0.85,
            colorbar_y=0.5,
            colorbar_bgcolor='rgba(255,255,255,0.5)'
        ),
        name="Field magnitude"
    ))

    fig.update_geos(
        projection_type="natural earth",
        landcolor="lightgrey",
        oceancolor="lightblue",
        showland=True,
        showocean=True,
        showcountries=True,
        showcoastlines=True,
        showframe=False,
        coastlinewidth=0.5,
        coastlinecolor="white"
    )

    fig.update_layout(
        title="Electric field magnitude during the orbit",
        geo=dict(
            showframe=False,
            showcoastlines=False,
            projection_type='natural earth'
        ),
        geo_scope='world',
        height=600,
        margin={"r":0,"t":30,"l":0,"b":0},
        showlegend=False
    )

    fig.show()

In [ ]:
def plot_EFD(path):
    f = xr.open_dataset(path, phony_dims='sort')
    X_Waveform = f['A111_W'][...]
    Y_Waveform = f['A112_W'][...]
    Z_Waveform = f['A113_W'][...]
    verse_time = f['VERSE_TIME'][...].values.flatten()
    X_Power_spectrum = f['A111_P'][...]
    Y_Power_spectrum = f['A112_P'][...]
    Z_Power_spectrum = f['A113_P'][...]
    latitude = f['MAG_LAT'][...]
    longitude = f['MAG_LON'][...]
    frequency = f['FREQ'][...]
    magnitude = np.sqrt(X_Waveform**2 + Y_Waveform**2 + Z_Waveform**2)
    polar_angle = np.arccos(Z_Waveform / magnitude)  # theta
    azimuthal_angle = np.arctan2(Y_Waveform, X_Waveform)  # phi
 
    # Convert angles to degrees
    polar_angle = np.degrees(polar_angle)
    azimuthal_angle = np.degrees(azimuthal_angle)
 
    reduced_freq = 100  # Specify the desired frequency here
    X_Waveform = reduce_frequency(X_Waveform, reduced_freq)
    Y_Waveform = reduce_frequency(Y_Waveform, reduced_freq)
    Z_Waveform = reduce_frequency(Z_Waveform, reduced_freq)
    magnitude = reduce_frequency(magnitude, reduced_freq)
    polar_angle = reduce_frequency(polar_angle, reduced_freq)
    azimuthal_angle = reduce_frequency(azimuthal_angle, reduced_freq)
 
    X_Waveform = X_Waveform.values.flatten()
    Y_Waveform = Y_Waveform.values.flatten()
    Z_Waveform = Z_Waveform.values.flatten()
    magnitude = magnitude.values.flatten()
    polar_angle = polar_angle.values.flatten()
    azimuthal_angle = azimuthal_angle.values.flatten()
 
    len_time = len(verse_time)
 
    # Ensure vers_extend has the same length as the waveforms
    vers_extend = np.concatenate([np.linspace(verse_time[i], verse_time[i+1], reduced_freq, endpoint=False) for i in range(len_time-1)])
    vers_extend = np.concatenate([vers_extend, np.linspace(verse_time[-2], verse_time[-1], reduced_freq)])
    # First figure for waveforms and magnitude
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=vers_extend, y=X_Waveform, mode='lines', name='X Waveform'))
    fig1.add_trace(go.Scatter(x=vers_extend, y=Y_Waveform, mode='lines', name='Y Waveform'))
    fig1.add_trace(go.Scatter(x=vers_extend, y=Z_Waveform, mode='lines', name='Z Waveform'))
    fig1.add_trace(go.Scatter(x=vers_extend, y=magnitude, mode='lines', name='Vector sum'))
 
    fig1.update_layout(
        title="X, Y, Z Waveforms and the Vector Sum",
        xaxis_title="Time (ms)",
        yaxis_title="V/m",
        legend=dict(x=1, y=0.5)
    )
 
    # Second figure for polar and azimuthal angles
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=vers_extend, y=polar_angle, mode='lines', name='Polar Angle', yaxis='y2'))
    fig2.add_trace(go.Scatter(x=vers_extend, y=azimuthal_angle, mode='lines', name='Azimuthal Angle', yaxis='y3'))
 
    fig2.update_layout(
        title="Polar and Azimuthal Angles",
        xaxis_title="Time (ms)",
        yaxis2=dict(
            title="Polar Angle (degrees)",
            range=[0, 180],
            side='left'
        ),
        yaxis3=dict(
            title="Azimuthal Angle (degrees)",
            range=[0, 360],
            overlaying='y2',
            side='right'
        ),
        legend=dict(x=1.08, y=0.54)
    )
 
    # Third figure for power spectra
    power_spectra = {'X': X_Power_spectrum, 'Y': Y_Power_spectrum, 'Z': Z_Power_spectrum}
    for axis, spectrum in power_spectra.items():
        fig = go.Figure()
        fig.add_trace(go.Heatmap(
            z=np.log10(spectrum.values.T),  # Apply log10 to intensity (z-axis)
            x=verse_time,
            y=frequency,
            colorscale='Viridis',
            colorbar=dict(title='Log Power Spectrum')
        ))
        fig.update_layout(
            title=f"{axis} Power Spectrum",
            xaxis_title="Time (ms)",
            yaxis_title="Frequency (Hz)",
            legend=dict(x=1, y=0.5)
        )
        fig.show()
 
    # Display the first two figures
    fig1.show()
    fig2.show()

In [ ]:
f = xr.open_dataset(EFD, phony_dims='sort')

latitude = f['MAG_LAT'][...]
longitude = f['MAG_LON'][...]
X_Waveform = f['A111_W'][...]
Y_Waveform = f['A112_W'][...]
Z_Waveform = f['A113_W'][...]
magnitude=  np.sqrt(X_Waveform**2 + Y_Waveform**2 + Z_Waveform**2)
plot_EFD_on_map(magnitude, latitude, longitude)
plot_EFD(EFD)

### Spectrogram

In [ ]:
def plot_spectrogram(z_data, x_data, y_data, title, x_title, y_title):
    fig = go.Figure(go.Heatmap(
        z=z_data.T,  # Transpose the data for correct orientation
        x=x_data,
        y=y_data,
        colorscale='Viridis',
        colorbar=dict(title='Particles/cm²/s/sr/MeV')
    ))

    fig.update_layout(
        title=title,
        xaxis_title=x_title,
        yaxis_title=y_title,
        legend=dict(x=1, y=0.5)
    )

    fig.show()


In [ ]:
def plot_variables(file_path):
    with xr.open_dataset(file_path, engine='h5netcdf', phony_dims='sort') as f:
        data = {
            'UTC_TIME': f['UTC_TIME'][:].values.flatten(),
            'A411': f['A411'][:].values,
            'A412': f['A412'][:].values,
            'Energy_Table_Electron': f['Energy_Table_Electron'][:].values,
            'Energy_Table_Proton': f['Energy_Table_Proton'][:].values
        }
    
    # Electron Energy Spectrum
    a411_data = np.reshape(data['A411'], (-1, 256, 9))
    a411_data_mean = np.mean(a411_data, axis=0)
    plot_spectrogram(a411_data_mean, data['UTC_TIME'], np.arange(256), 
                     "Electron Energy Spectrum", "Time (UTC)", "Energy (MeV)")

    # Electron Pitch Angle Spectrum
    a411_pitch_angle_mean = np.mean(a411_data, axis=1)
    plot_spectrogram(a411_pitch_angle_mean, data['UTC_TIME'], np.arange(9), 
                     "Electron Pitch Angle Spectrum", "Time (UTC)", "Pitch Angle (degrees)")

    # Proton Energy Spectrum
    a412_data = np.reshape(data['A412'], (-1, 256, 9))
    a412_data_mean = np.mean(a412_data, axis=0)
    plot_spectrogram(a412_data_mean, data['UTC_TIME'], np.arange(256), 
                     "Proton Energy Spectrum", "Time (UTC)", "Energy (MeV)")

    # Proton Pitch Angle Spectrum
    a412_pitch_angle_mean = np.mean(a412_data, axis=1)
    plot_spectrogram(a412_pitch_angle_mean, data['UTC_TIME'], np.arange(9), 
                     "Proton Pitch Angle Spectrum", "Time (UTC)", "Pitch Angle (degrees)")

In [21]:
def plot_twin_timeline_verse_time(fig, path):
    # Open the dataset using the h5netcdf engine
    f = xr.open_dataset(path, engine='h5netcdf', phony_dims='sort')
    
    # Extract the required variables
    verse_time = f['VERSE_TIME'][...].values.flatten()
    data = f['A311'][...]
    data2 = f['A321'][...]
    
    # Reduce the frequency of the data
    data = reduce_frequency(data, 1)
    data2 = reduce_frequency(data2, 1)
    
    log = True
    
    # Get the frequency dimension
    try:
        freq = data.shape[1]
    except:
        freq = 1
    
    # Remove the first element of the data (it sometimes gives weird values) and flatten it to be able to plot it
    data = data.values[1:].flatten()
    data2 = data2.values[1:].flatten()
    
    # Remove the first element of the verse time and flatten it
    verse_time = verse_time[1:]
    
    # Get the length to be able to plot it
    len_time = len(verse_time)
    
    # Plot everything
    vers_extend = np.concatenate([np.linspace(verse_time[i], verse_time[i+1], freq, endpoint=False) for i in range(len_time-1)])
    vers_extend = np.concatenate([vers_extend, np.linspace(verse_time[-2], verse_time[-1], freq)])
    
    # Create subplots with a secondary y-axis
    fig.add_trace(
        go.Scatter(x=vers_extend, y=data, name="Electron Density", line=dict(color='blue')),
        secondary_y=False
    )
    
    fig.add_trace(
        go.Scatter(x=vers_extend, y=data2, name="Electron Temperature", line=dict(color='red')),
        secondary_y=True  # y-axis on RHS
    )
    
    # Configure y-axes
    fig.update_yaxes(title_text="1/m^3", secondary_y=False)
    fig.update_yaxes(title_text="K", secondary_y=True)
    if log:
        fig.update_yaxes(type="log", secondary_y=False)
        fig.update_yaxes(type="log", secondary_y=True)
    
    # Configure x-axis
    fig.update_xaxes(title_text="ms")
    
    return fig

In [ ]:
def plot_twin_timeline_utc(fig, path):
    f = xr.open_dataset(path, engine='h5netcdf', phony_dims='sort')
    
    # Extract the required variables
    time = f['UTC_TIME'][...].values.flatten()
    data = f['A311'][...]
    data2 = f['A321'][...]
    
    # Reduce the frequency of the data
    data = reduce_frequency(data, 1)
    data2 = reduce_frequency(data2, 1)
    
    log = True

    def convert_to_utc_time(date_strings):
        utc_times = pd.to_datetime(date_strings, format="%Y%m%d%H%M%S%f", utc=True)
        return utc_times

    # Catch all problems with frequency
    try:
        freq = data.shape[1]
    except:
        freq = 1
    
    # Remove the first element of the data and flatten it to be able to plot it
    data = data.values[1:].flatten()
    data2 = data2.values[1:].flatten()
    
    # Remove the first element of the time and flatten it
    time = time[1:]
    
    len_time = len(time)
    time_extend = np.concatenate([np.linspace(time[i], time[i+1], freq, endpoint=False) for i in range(len_time - 1)])
    time_extend = np.concatenate([time_extend, np.linspace(time[-2], time[-1], freq)])
    utctimes = convert_to_utc_time(time_extend)
    
    df = pd.DataFrame({
        "data": data,
        "data2": data2,
        "time": utctimes
    })
    
    # Plotting with Plotly
    fig.add_trace(go.Scatter(x=df['time'], y=df['data'], mode='lines', name='Electron Density'), secondary_y=False)
    fig.add_trace(go.Scatter(x=df['time'], y=df['data2'], mode='lines', name='Electron Temperature', line=dict(color='red')), secondary_y=True)

    # Update y-axes titles
    fig.update_yaxes(title_text="1/m^3", secondary_y=False)
    fig.update_yaxes(title_text="K", secondary_y=True)
    if log:
        fig.update_yaxes(type="log", secondary_y=False)
        fig.update_yaxes(type="log", secondary_y=True)

    # Update x-axis title
    fig.update_xaxes(title_text="Time (UTC)")

    return fig

In [ ]:
def plot_on_map_density(fig, path):
    f = xr.open_dataset(path, engine='h5netcdf', phony_dims='sort')
    
    # Extract the required variables
    latitude = f['GEO_LAT'][...]
    longitude = f['GEO_LON'][...]
    data = f['A311'][...]

    # Reduce the frequency of the data
    data = reduce_frequency(data, 1)

    try:
        freq = data.shape[1]
    except:
        freq = 1

    measure = data.values[1:].flatten()
    lon = longitude.values[1:].flatten()
    lat = latitude.values[1:].flatten()
    
    length_coord = len(lon)
    length_measure = len(measure)

    lon_extend = np.concatenate([np.linspace(lon[i], lon[i+1], freq, endpoint=False) for i in range(length_coord-1)])
    lon_extend = np.concatenate([lon_extend, np.linspace(lon[-2], lon[-1], freq)])
    
    lat_extend = np.concatenate([np.linspace(lat[i], lat[i+1], freq, endpoint=False) for i in range(length_coord-1)])
    lat_extend = np.concatenate([lat_extend, np.linspace(lat[-2], lat[-1], freq)])

    scatter = go.Scattergeo(
        lon=lon_extend,
        lat=lat_extend,
        text=measure,
        marker=dict(
            size=10,
            color=measure,
            colorscale='Viridis',
            colorbar=dict(title="1/m^3"),
            cmin=measure.min(),
            cmax=10**11,
            showscale=True
        ),
        mode='markers'
    )
    
    fig.add_trace(scatter)

    # Update the layout
    fig.update_layout(
        geo=dict(
            showland=True,
            landcolor="lightgrey",
        ),
        autosize=False,
        width=800,
        height=600,
    )
    fig.update_layout(title="Electron Density", template="plotly_white")

    return fig


In [ ]:
def plot_on_map_temperature(fig, path):
    f = xr.open_dataset(path, engine='h5netcdf', phony_dims='sort')
    
    # Extract the required variables
    latitude = f['GEO_LAT'][...]
    longitude = f['GEO_LON'][...]
    data = f['A321'][...]

    # Reduce the frequency of the data
    data = reduce_frequency(data, 1)

    try:
        freq = data.shape[1]
    except:
        freq = 1

    measure = data.values[1:].flatten()
    lon = longitude.values[1:].flatten()
    lat = latitude.values[1:].flatten()
    
    length_coord = len(lon)
    length_measure = len(measure)

    lon_extend = np.concatenate([np.linspace(lon[i], lon[i+1], freq, endpoint=False) for i in range(length_coord-1)])
    lon_extend = np.concatenate([lon_extend, np.linspace(lon[-2], lon[-1], freq)])
    
    lat_extend = np.concatenate([np.linspace(lat[i], lat[i+1], freq, endpoint=False) for i in range(length_coord-1)])
    lat_extend = np.concatenate([lat_extend, np.linspace(lat[-2], lat[-1], freq)])

    scatter = go.Scattergeo(
        lon=lon_extend,
        lat=lat_extend,
        text=measure,
        marker=dict(
            size=10,
            color=measure,
            colorscale='Viridis',
            colorbar=dict(title="K"),
            cmin=1000,
            cmax=3000,
            showscale=True
        ),
        mode='markers'
    )

    fig.add_trace(scatter)

    # Update the layout
    fig.update_layout(
        geo=dict(
            showland=True,
            landcolor="lightgrey",
        ),
        autosize=False,
        width=800,
        height=600,
    )
    fig.update_layout(title="Electron Temperature", template="plotly_white")

    return fig


In [ ]:
def lap_plot(f_path):
    # Create a 2x2 grid for plotting
    columns = st.columns(2)
    
    # Plot Electron Temperature and Electron Density on the first row
    with columns[0]:
        fig1 = make_subplots(specs=[[{"secondary_y": True}]])
        plot_twin_timeline_utc(fig1, f_path)
        st.plotly_chart(fig1)

    with columns[1]:
        fig2 = make_subplots(specs=[[{"secondary_y": True}]])
        plot_twin_timeline_verse_time(fig2, f_path)
        st.plotly_chart(fig2)

    # Plot Electron Density and Electron Temperature on the second row
    with columns[0]:
        fig3 = go.Figure()
        plot_on_map_density(fig3, f_path)
        st.plotly_chart(fig3)

    with columns[1]:
        fig4 = go.Figure()
        plot_on_map_temperature(fig4, f_path)
        st.plotly_chart(fig4)

In [ ]:
def plot_EFD_on_map(data):
    f = xr.open_dataset(data, engine='h5netcdf', phony_dims='sort')
    latitude = f['MAG_LAT'][...]
    longitude = f['MAG_LON'][...]
    try:
        freq = data.shape[1]
    except IndexError:
        freq = 1

    measure = data.values.flatten()

    lon = longitude.values.flatten()
    lat = latitude.values.flatten()

    length_coord = len(lon)
    length_measure = len(measure)

    lon_extend = np.concatenate([np.linspace(lon[i], lon[i+1], freq, endpoint=False) for i in range(length_coord-1)])
    lon_extend = np.concatenate([lon_extend, np.linspace(lon[-2], lon[-1], freq)])

    lat_extend = np.concatenate([np.linspace(lat[i], lat[i+1], freq, endpoint=False) for i in range(length_coord-1)])
    lat_extend = np.concatenate([lat_extend, np.linspace(lat[-2], lat[-1], freq)])

    # Create a scatter plot on a map using Plotly
    fig = go.Figure()

    # Load world map using geopandas
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

    # Plot world map as background
    fig.add_trace(go.Choropleth(
        geojson=world.__geo_interface__,
        locations=world.index,
        z=np.zeros(len(world)),  # Dummy values for Choropleth trace
        colorscale=[[0, 'lightgrey'], [1, 'lightgrey']],
        hoverinfo='none',
        showscale=False
    ))

    # Scatter plot for data points
    fig.add_trace(go.Scattergeo(
        lon=lon_extend,
        lat=lat_extend,
        mode='markers',
        marker=dict(
            size=10,
            color=measure,
            colorscale='Viridis',
            colorbar=dict(title="V/m"),
            opacity=0.8,
            colorbar_thickness=20,
            colorbar_x=0.85,
            colorbar_y=0.5,
            colorbar_bgcolor='rgba(255,255,255,0.5)'
        ),
        name="Field magnitude"
    ))

    fig.update_geos(
        projection_type="natural earth",
        landcolor="lightgrey",
        oceancolor="lightblue",
        showland=True,
        showocean=True,
        showcountries=True,
        showcoastlines=True,
        showframe=False,
        coastlinewidth=0.5,
        coastlinecolor="white"
    )

    fig.update_layout(
        title="Electric field magnitude during the orbit",
        geo=dict(
            showframe=False,
            showcoastlines=False,
            projection_type='natural earth'
        ),
        geo_scope='world',
        height=600,
        margin={"r":0,"t":30,"l":0,"b":0},
        showlegend=False
    )
    st.plotly_chart(fig)

In [ ]:
def reduce_frequency(data_array, frequency):
    """
    Reduce the number of measurements in each row of the xarray based on the specified frequency,
    while preserving the metadata.
    Parameters:
    data_array (xarray.DataArray): The input xarray.
    frequency (int): The number of elements to keep in each row.
    Returns:
    xarray.DataArray: The reduced xarray.
    """
    num_rows, num_cols = data_array.shape
    if frequency > num_cols:
        raise ValueError("Frequency cannot be greater than the number of columns in the data array.")
    reduced_rows = []
    for row in data_array.values:
        indices = np.linspace(0, num_cols - 1, frequency, dtype=int)
        reduced_row = row[indices]
        reduced_rows.append(reduced_row)
    reduced_values = np.stack(reduced_rows, axis=0)
    reduced_data_array = xr.DataArray(reduced_values,
                                      dims=[data_array.dims[0], 'reduced_freq'],
                                      coords={data_array.dims[0]: data_array.coords[data_array.dims[0]],
                                              'reduced_freq': indices},
                                      attrs=data_array.attrs)
    return reduced_data_array

In [ ]:
def plot_SCM(path):
    f = xr.open_dataset(path, engine='h5netcdf', phony_dims='sort')
    X_Waveform = f['A231_W'][...]
    Y_Waveform = f['A232_W'][...]
    Z_Waveform = f['A233_W'][...]
    verse_time = f['VERSE_TIME'][...].values.flatten()
    X_Power_spectrum = f['A231_P'][...]
    Y_Power_spectrum = f['A232_P'][...]
    Z_Power_spectrum = f['A233_P'][...]
    latitude = f['MAG_LAT'][...]
    longitude = f['MAG_LON'][...]
    frequency = f['FREQ'][...]
    magnitude = np.sqrt(X_Waveform**2 + Y_Waveform**2 + Z_Waveform**2)
    polar_angle = np.arccos(Z_Waveform / magnitude)  # theta
    azimuthal_angle = np.arctan2(Y_Waveform, X_Waveform)  # phi
 
    # Convert angles to degrees
    polar_angle = np.degrees(polar_angle)
    azimuthal_angle = np.degrees(azimuthal_angle)
 
    reduced_freq = 100  # Specify the desired frequency here
    X_Waveform = reduce_frequency(X_Waveform, reduced_freq)
    Y_Waveform = reduce_frequency(Y_Waveform, reduced_freq)
    Z_Waveform = reduce_frequency(Z_Waveform, reduced_freq)
    magnitude = reduce_frequency(magnitude, reduced_freq)
    polar_angle = reduce_frequency(polar_angle, reduced_freq)
    azimuthal_angle = reduce_frequency(azimuthal_angle, reduced_freq)
 
    X_Waveform = X_Waveform.values.flatten()
    Y_Waveform = Y_Waveform.values.flatten()
    Z_Waveform = Z_Waveform.values.flatten()
    magnitude = magnitude.values.flatten()
    polar_angle = polar_angle.values.flatten()
    azimuthal_angle = azimuthal_angle.values.flatten()
 
    len_time = len(verse_time)
 
    # Ensure vers_extend has the same length as the waveforms
    vers_extend = np.concatenate([np.linspace(verse_time[i], verse_time[i+1], reduced_freq, endpoint=False) for i in range(len_time-1)])
    vers_extend = np.concatenate([vers_extend, np.linspace(verse_time[-2], verse_time[-1], reduced_freq)])
    # First figure for waveforms and magnitude
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=vers_extend, y=X_Waveform, mode='lines', name='X Waveform'))
    fig1.add_trace(go.Scatter(x=vers_extend, y=Y_Waveform, mode='lines', name='Y Waveform'))
    fig1.add_trace(go.Scatter(x=vers_extend, y=Z_Waveform, mode='lines', name='Z Waveform'))
    fig1.add_trace(go.Scatter(x=vers_extend, y=magnitude, mode='lines', name='Vector sum'))
 
    fig1.update_layout(
        title="X, Y, Z Waveforms and the Vector Sum",
        xaxis_title="Time (ms)",
        yaxis_title="V/m",
        legend=dict(x=1, y=0.5)
    )
 
    # Second figure for polar and azimuthal angles
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=vers_extend, y=polar_angle, mode='lines', name='Polar Angle', yaxis='y2'))
    fig2.add_trace(go.Scatter(x=vers_extend, y=azimuthal_angle, mode='lines', name='Azimuthal Angle', yaxis='y3'))
 
    fig2.update_layout(
        title="Polar and Azimuthal Angles",
        xaxis_title="Time (ms)",
        yaxis2=dict(
            title="Polar Angle (degrees)",
            range=[0, 180],
            side='left'
        ),
        yaxis3=dict(
            title="Azimuthal Angle (degrees)",
            range=[0, 360],
            overlaying='y2',
            side='right'
        ),
        legend=dict(x=1.08, y=0.54)
    )
 
    # Third figure for power spectra
    power_spectra = {'X': X_Power_spectrum, 'Y': Y_Power_spectrum, 'Z': Z_Power_spectrum}
    for axis, spectrum in power_spectra.items():
        fig = go.Figure()
        fig.add_trace(go.Heatmap(
            z=np.log10(spectrum.values.T),  # Apply log10 to intensity (z-axis)
            x=verse_time,
            y=frequency,
            colorscale='Viridis',
            colorbar=dict(title='Log Power Spectrum')
        ))
        fig.update_layout(
            title=f"{axis} Power Spectrum",
            xaxis_title="Time (ms)",
            yaxis_title="Frequency (Hz)",
            legend=dict(x=1, y=0.5)
        )
        st.plotly_chart(fig1)
 
    # Display the first two figures
    st.plotly_chart(fig2)
    


# def plot_SCM_on_map(data):
#     f = xr.open_dataset(data, engine='h5netcdf', phony_dims='sort')
#     latitude = f['MAG_LAT'][...]
#     longitude = f['MAG_LON'][...]
#     try:
#         freq = data.shape[1]
#     except IndexError:
#         freq = 1

#     measure = data.values.flatten()

#     lon = longitude.values.flatten()
#     lat = latitude.values.flatten()

#     length_coord = len(lon)
#     length_measure = len(measure)

#     lon_extend = np.concatenate([np.linspace(lon[i], lon[i+1], freq, endpoint=False) for i in range(length_coord-1)])
#     lon_extend = np.concatenate([lon_extend, np.linspace(lon[-2], lon[-1], freq)])

#     lat_extend = np.concatenate([np.linspace(lat[i], lat[i+1], freq, endpoint=False) for i in range(length_coord-1)])
#     lat_extend = np.concatenate([lat_extend, np.linspace(lat[-2], lat[-1], freq)])

#     # Create a scatter plot on a map using Plotly
#     fig = go.Figure()

#     # Load world map using geopandas
#     world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

#     # Plot world map as background
#     fig.add_trace(go.Choropleth(
#         geojson=world.__geo_interface__,
#         locations=world.index,
#         z=np.zeros(len(world)),  # Dummy values for Choropleth trace
#         colorscale=[[0, 'lightgrey'], [1, 'lightgrey']],
#         hoverinfo='none',
#         showscale=False
#     ))

#     # Scatter plot for data points
#     fig.add_trace(go.Scattergeo(
#         lon=lon_extend,
#         lat=lat_extend,
#         mode='markers',
#         marker=dict(
#             size=10,
#             color=measure,
#             colorscale='Viridis',
#             colorbar=dict(title="V/m"),
#             opacity=0.8,
#             colorbar_thickness=20,
#             colorbar_x=0.85,
#             colorbar_y=0.5,
#             colorbar_bgcolor='rgba(255,255,255,0.5)'
#         ),
#         name="Field magnitude"
#     ))

#     fig.update_geos(
#         projection_type="natural earth",
#         landcolor="lightgrey",
#         oceancolor="lightblue",
#         showland=True,
#         showocean=True,
#         showcountries=True,
#         showcoastlines=True,
#         showframe=False,
#         coastlinewidth=0.5,
#         coastlinecolor="white"
#     )

#     fig.update_layout(
#         title="Electric field magnitude during the orbit",
#         geo=dict(
#             showframe=False,
#             showcoastlines=False,
#             projection_type='natural earth'
#         ),
#         geo_scope='world',
#         height=600,
#         margin={"r":0,"t":30,"l":0,"b":0},
#         showlegend=False
#     )

#     fig.show()

# def plot_SCM_on_map(data, latitude, longitude):
    
#     try:
#         freq = data.shape[1]
#     except IndexError:
#         freq = 1

#     measure = data.values.flatten()

#     lon = longitude.values.flatten()
#     lat = latitude.values.flatten()

#     length_coord = len(lon)
#     length_measure = len(measure)

#     lon_extend = np.concatenate([np.linspace(lon[i], lon[i+1], freq, endpoint=False) for i in range(length_coord-1)])
#     lon_extend = np.concatenate([lon_extend, np.linspace(lon[-2], lon[-1], freq)])

#     lat_extend = np.concatenate([np.linspace(lat[i], lat[i+1], freq, endpoint=False) for i in range(length_coord-1)])
#     lat_extend = np.concatenate([lat_extend, np.linspace(lat[-2], lat[-1], freq)])

#     # Create a scatter plot on a map using Plotly
#     fig = go.Figure()

#     # Load world map from the local directory
#     world = gpd.read_file('ne_110m_admin_0_countries.shp')

#     # Plot world map as background
#     fig.add_trace(go.Choropleth(
#         geojson=world.__geo_interface__,
#         locations=world.index,
#         z=np.zeros(len(world)),  # Dummy values for Choropleth trace
#         colorscale=[[0, 'lightgrey'], [1, 'lightgrey']],
#         hoverinfo='none',
#         showscale=False
#     ))

#     # Scatter plot for data points
#     fig.add_trace(go.Scattergeo(
#         lon=lon_extend,
#         lat=lat_extend,
#         mode='markers',
#         marker=dict(
#             size=10,
#             color=measure,
#             colorscale='Viridis',
#             colorbar=dict(title="V/m"),
#             opacity=0.8,
#             colorbar_thickness=20,
#             colorbar_x=0.85,
#             colorbar_y=0.5,
#             colorbar_bgcolor='rgba(255,255,255,0.5)'
#         ),
#         name="Field magnitude"
#     ))

#     fig.update_geos(
#         projection_type="natural earth",
#         landcolor="lightgrey",
#         oceancolor="lightblue",
#         showland=True,
#         showocean=True,
#         showcountries=True,
#         showcoastlines=True,
#         showframe=False,
#         coastlinewidth=0.5,
#         coastlinecolor="white"
#     )

#     fig.update_layout(
#         title="Electric field magnitude during the orbit",
#         geo=dict(
#             showframe=False,
#             showcoastlines=False,
#             projection_type='natural earth'
#         ),
#         geo_scope='world',
#         height=600,
#         margin={"r":0,"t":30,"l":0,"b":0},
#         showlegend=False
#     )
#     st.plotly_chart(fig)

def plot_SCM_on_map(data, latitude, longitude, downsample_factor=50):
    try:
        freq = data.shape[1]
    except IndexError:
        freq = 1

    measure = data.values.flatten()
    lon = longitude.values.flatten()
    lat = latitude.values.flatten()

    # Downsample the data
    measure = measure[::downsample_factor]
    lon = lon[::downsample_factor]
    lat = lat[::downsample_factor]

    length_coord = len(lon)
    length_measure = len(measure)

    lon_extend = np.concatenate([np.linspace(lon[i], lon[i+1], freq, endpoint=False) for i in range(length_coord-1)])
    lon_extend = np.concatenate([lon_extend, np.linspace(lon[-2], lon[-1], freq)])

    lat_extend = np.concatenate([np.linspace(lat[i], lat[i+1], freq, endpoint=False) for i in range(length_coord-1)])
    lat_extend = np.concatenate([lat_extend, np.linspace(lat[-2], lat[-1], freq)])

    # Downsample the extended coordinates
    lon_extend = lon_extend[::downsample_factor]
    lat_extend = lat_extend[::downsample_factor]

    # Create a scatter plot on a map using Plotly
    fig = go.Figure()

    # Load world map from the local directory
    world = gpd.read_file('Earth_Data/ne_110m_admin_0_countries.shp')

    # Plot world map as background
    fig.add_trace(go.Choropleth(
        geojson=world.__geo_interface__,
        locations=world.index,
        z=np.zeros(len(world)),  # Dummy values for Choropleth trace
        colorscale=[[0, 'lightgrey'], [1, 'lightgrey']],
        hoverinfo='none',
        showscale=False
    ))

    # Scatter plot for data points
    fig.add_trace(go.Scattergeo(
        lon=lon_extend,
        lat=lat_extend,
        mode='markers',
        marker=dict(
            size=10,
            color=measure,
            colorscale='Viridis',
            colorbar=dict(title="V/m"),
            opacity=0.8,
            colorbar_thickness=20,
            colorbar_x=0.85,
            colorbar_y=0.5,
            colorbar_bgcolor='rgba(255,255,255,0.5)'
        ),
        name="Field magnitude"
    ))

    fig.update_geos(
        projection_type="natural earth",
        landcolor="lightgrey",
        oceancolor="lightblue",
        showland=True,
        showocean=True,
        showcountries=True,
        showcoastlines=True,
        showframe=False,
        coastlinewidth=0.5,
        coastlinecolor="white"
    )

    fig.update_layout(
        title="Electric field magnitude during the orbit",
        geo=dict(
            showframe=False,
            showcoastlines=False,
            projection_type='natural earth'
        ),
        geo_scope='world',
        height=600,
        margin={"r":0,"t":30,"l":0,"b":0},
        showlegend=False
    )
    st.plotly_chart(fig)

def scmplot(file_path):
    f = xr.open_dataset(file_path, engine='h5netcdf', phony_dims='sort')
    latitude = f['MAG_LAT'][...]
    longitude = f['MAG_LON'][...]
    X_Waveform = f['A231_W'][...]
    Y_Waveform = f['A232_W'][...]
    Z_Waveform = f['A233_W'][...]
    magnitude=  np.sqrt(X_Waveform**2 + Y_Waveform**2 + Z_Waveform**2)
    plot_SCM_on_map(magnitude, latitude, longitude)
    plot_SCM(file_path)
    # plot_EFD_on_map(file_path)